In [6]:
# Libraries import karo
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import plotly.graph_objects as go
import plotly.io as pio
import os

pio.renderers.default = 'notebook_connected'

# CSV se data load karo
df = pd.read_csv(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\data\reliance_featured_data.csv',
                 index_col='Date',
                 parse_dates=True)

print("Data loaded! ✅")
print("Shape:", df.shape)

Data loaded! ✅
Shape: (1335, 30)


In [5]:
!pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Target variable banao
df['Target'] = df['Close'].shift(-1)
df = df.dropna()

# Features aur Target alag karo
X = df.drop(columns=['Target'])
y = df['Target']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

# Scaling
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

print("Preprocessing complete! ✅")
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows:  {X_test.shape[0]}")

Preprocessing complete! ✅
Training rows: 1067
Testing rows:  267


LINEAR REGRESSION

In [8]:
# Model 1 - Linear Regression
print("Model 1 - Linear Regression Training...")
print("=" * 40)

# Model banao
lr_model = LinearRegression()

# Training data pe train karo
lr_model.fit(X_train_scaled, y_train_scaled)

print("Linear Regression trained! ✅")

# Test data pe predict karo
lr_pred_scaled = lr_model.predict(X_test_scaled)

# Predictions ko original scale pe wapas lao
lr_pred = scaler_y.inverse_transform(lr_pred_scaled.reshape(-1, 1)).ravel()

# Evaluation
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_r2   = r2_score(y_test, lr_pred)

print(f"\nLinear Regression Results:")
print(f"RMSE: ₹{lr_rmse:.2f}")
print(f"MAE:  ₹{lr_mae:.2f}")
print(f"R²:   {lr_r2:.4f}")

Model 1 - Linear Regression Training...
Linear Regression trained! ✅

Linear Regression Results:
RMSE: ₹18.94
MAE:  ₹14.22
R²:   0.9564


RANDOM FOREST

In [9]:
# Model 2 - Random Forest
print("Model 2 - Random Forest Training...")
print("=" * 40)
print("Thoda time lagega... ⏳")

# Model banao
rf_model = RandomForestRegressor(
    n_estimators=100,    # 100 trees
    random_state=42,     # reproducibility ke liye
    n_jobs=-1            # saare CPU cores use karo
)

# Train karo
rf_model.fit(X_train_scaled, y_train_scaled)

print("Random Forest trained! ✅")

# Predict karo
rf_pred_scaled = rf_model.predict(X_test_scaled)

# Original scale pe wapas lao
rf_pred = scaler_y.inverse_transform(rf_pred_scaled.reshape(-1, 1)).ravel()

# Evaluation
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)

print(f"\nRandom Forest Results:")
print(f"RMSE: ₹{rf_rmse:.2f}")
print(f"MAE:  ₹{rf_mae:.2f}")
print(f"R²:   {rf_r2:.4f}")

Model 2 - Random Forest Training...
Thoda time lagega... ⏳
Random Forest trained! ✅

Random Forest Results:
RMSE: ₹24.86
MAE:  ₹19.63
R²:   0.9248


XGBOOST

XGBoost = Extreme Gradient Boosting
Random Forest mein 100 trees parallel kaam karte hain!
XGBoost mein trees series mein kaam karte hain — har tree pehle tree ki galti sudharta hai!
Isliye ye sabse powerful model hai! 💪

In [10]:
# Model 3 - XGBoost
print("Model 3 - XGBoost Training...")
print("=" * 40)

# Model banao
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

# Train karo
xgb_model.fit(
    X_train_scaled, y_train_scaled,
    eval_set=[(X_test_scaled, y_test_scaled)],
    verbose=False
)

print("XGBoost trained! ✅")

# Predict karo
xgb_pred_scaled = xgb_model.predict(X_test_scaled)

# Original scale pe wapas lao
xgb_pred = scaler_y.inverse_transform(xgb_pred_scaled.reshape(-1, 1)).ravel()

# Evaluation
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_r2   = r2_score(y_test, xgb_pred)

print(f"\nXGBoost Results:")
print(f"RMSE: ₹{xgb_rmse:.2f}")
print(f"MAE:  ₹{xgb_mae:.2f}")
print(f"R²:   {xgb_r2:.4f}")

Model 3 - XGBoost Training...
XGBoost trained! ✅

XGBoost Results:
RMSE: ₹28.80
MAE:  ₹23.08
R²:   0.8991


💡 Linear Regression Best Kyun Raha?

Stock price data mostly linear trend follow karta hai — upar jaati rehti hai gradually!
Complex models jaise XGBoost aur Random Forest non-linear patterns dhundhte hain — jo yahan zyada nahi hain!
Isliye simple model ne better kaam kiya — ye Data Science mein common hota hai! 😊
"Simple models often beat complex ones!" — ye ek important lesson hai!

In [11]:
# Models save karo
models_path = r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\models'

# Linear Regression save
with open(f'{models_path}\\lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

# Random Forest save
with open(f'{models_path}\\rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# XGBoost save
with open(f'{models_path}\\xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("Teeno models save ho gaye! ✅")
print(f"Location: models/ folder")

Teeno models save ho gaye! ✅
Location: models/ folder


In [14]:
# Predicted vs Actual Chart
fig = go.Figure()

# Actual prices
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=y_test,
    name='Actual Price',
    line=dict(color='white', width=2)
))

# Linear Regression predictions
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=lr_pred,
    name='Linear Regression',
    line=dict(color='#FF6B6B', width=1.5)
))

# Random Forest predictions
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=rf_pred,
    name='Random Forest',
    line=dict(color='#4ECDC4', width=1.5)
))

# XGBoost predictions
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=xgb_pred,
    name='XGBoost',
    line=dict(color='#FFE66D', width=1.5)
))

fig.update_layout(
    title='Reliance Industries — Actual vs Predicted Price (2025-2026)',
    xaxis_title='Date',
    yaxis_title='Price (₹)',
    template='plotly_dark',
    height=500,
    hovermode='x unified'
)

fig.show()

In [13]:
!pip install kaleido


   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---- ----------------------------------- 1/9 [pluggy]
   ---- ----------------------------------- 1/9 [pluggy]
   -------- ------------------------------- 2/9 [orjson]
   ----------------- ---------------------- 4/9 [iniconfig]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import os

# Reports/charts folder banao
os.makedirs(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\stock_prediction_project\reports\charts', exist_ok=True)

# Predicted vs Actual Chart
fig = go.Figure()

# Actual prices
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=y_test,
    name='Actual Price',
    line=dict(color='white', width=2)
))

# Linear Regression predictions
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=lr_pred,
    name='Linear Regression',
    line=dict(color='#FF6B6B', width=1.5)
))

# Random Forest predictions
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=rf_pred,
    name='Random Forest',
    line=dict(color='#4ECDC4', width=1.5)
))

# XGBoost predictions
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=xgb_pred,
    name='XGBoost',
    line=dict(color='#FFE66D', width=1.5)
))

fig.update_layout(
    title='Reliance Industries — Actual vs Predicted Price (2025-2026)',
    xaxis_title='Date',
    yaxis_title='Price (₹)',
    template='plotly_dark',
    height=500,
    hovermode='x unified'
)

# Chart dikhao
fig.show()

# Chart save karo
fig.write_image(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\stock_prediction_project\reports\charts\predicted_vs_actual.png')
print("Chart saved! ✅")

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


💡 Business Insight #11:

Teeno models ne 2025-2026 ka trend sahi pakda! — March 2025 mein low, July 2025 mein high, phir correction — ye real market movement tha aur models ne predict kiya! Ye recruiter ko bahut impress karega! 😎

In [22]:
# Model Comparison Bar Chart
models = ['Linear Regression', 'Random Forest', 'XGBoost']
rmse_scores = [lr_rmse, rf_rmse, xgb_rmse]
mae_scores = [lr_mae, rf_mae, xgb_mae]
r2_scores = [lr_r2, rf_r2, xgb_r2]

from plotly.subplots import make_subplots

fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=('RMSE (Lower is Better)',
                                    'MAE (Lower is Better)',
                                    'R² Score (Higher is Better)'))

# RMSE
fig2.add_trace(go.Bar(
    x=models, y=rmse_scores,
    marker_color=['#26a69a', '#ef5350', '#FFE66D'],
    name='RMSE'
), row=1, col=1)

# MAE
fig2.add_trace(go.Bar(
    x=models, y=mae_scores,
    marker_color=['#26a69a', '#ef5350', '#FFE66D'],
    name='MAE'
), row=1, col=2)

# R2
fig2.add_trace(go.Bar(
    x=models, y=r2_scores,
    marker_color=['#26a69a', '#ef5350', '#FFE66D'],
    name='R²'
), row=1, col=3)

fig2.update_layout(
    title='Model Comparison — RMSE, MAE, R²',
    template='plotly_dark',
    height=400,
    showlegend=False
)

fig2.show()

# Save karo
fig2.write_image(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\stock_prediction_project\reports\charts\model_comparison.png')
print("Chart saved! ✅")

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [19]:
!pip install --upgrade kaleido


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from plotly.subplots import make_subplots

# Model Comparison Bar Chart
models = ['Linear Regression', 'Random Forest', 'XGBoost']
rmse_scores = [lr_rmse, rf_rmse, xgb_rmse]
mae_scores = [lr_mae, rf_mae, xgb_mae]
r2_scores = [lr_r2, rf_r2, xgb_r2]

fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=('RMSE (Lower is Better)',
                                    'MAE (Lower is Better)',
                                    'R² Score (Higher is Better)'))

# RMSE
fig2.add_trace(go.Bar(
    x=models, y=rmse_scores,
    marker_color=['#26a69a', '#ef5350', '#FFE66D'],
    name='RMSE'
), row=1, col=1)

# MAE
fig2.add_trace(go.Bar(
    x=models, y=mae_scores,
    marker_color=['#26a69a', '#ef5350', '#FFE66D'],
    name='MAE'
), row=1, col=2)

# R2
fig2.add_trace(go.Bar(
    x=models, y=r2_scores,
    marker_color=['#26a69a', '#ef5350', '#FFE66D'],
    name='R²'
), row=1, col=3)

fig2.update_layout(
    title='Model Comparison — RMSE, MAE, R²',
    template='plotly_dark',
    height=400,
    showlegend=False
)

fig2.show()

💡 Conclusion:

Linear Regression — hamare best model hai! 😎
Isko hum Deployment mein use karenge!

# Day 7 - Model Building & Training

## Models Trained

### 1. Linear Regression (Baseline)
- RMSE: ₹18.94
- MAE:  ₹14.22
- R²:   0.9564 ← BEST MODEL 🏆

### 2. Random Forest
- n_estimators: 100
- RMSE: ₹24.86
- MAE:  ₹19.63
- R²:   0.9248

### 3. XGBoost
- n_estimators: 100, learning_rate: 0.1
- RMSE: ₹28.80
- MAE:  ₹23.08
- R²:   0.8991

## Conclusion
- Linear Regression sabse best model raha
- Stock price linear trend follow karti hai
- Teeno models models/ folder mein save kiye

## Charts
- predicted_vs_actual.png
- model_comparison.png